In [ ]:
#@title
# import stuff

import numpy as np                            # numerical arrays and math functions
import pandas as pd                           # data tables (dataframes)
import matplotlib.pyplot as plt               # plotting
import statsmodels.api as sm                  # statistical models: OLS, Logit, etc.
from IPython.display import display, Markdown # render formatted text inside the notebook

# Notebook on Regression Methods

This notebook covers a few exercises in regression: first linear regression and then logistic regression. 

**How to use the Notebook**. The notebook is made so that by default all the code is hidden. If you just run, using the PLAY button, everthing sequentially, results will be produced, plots will be shown and explanations will appear. If you want to see the actual code that generates the results and plots, you can click on the bar at the left of the cell. Clicking a second time will hide the code again.

# What is regression?

When we do regression, we make a predictive model of one variable (the **response** or **preditction target**) using other variables, called **predictors** or **regressors**, which can be continuous or categorical/discrete. 

## Linear regression for continuous resposes 

The simplest model is a linear model, which is a good first start when the target is continuous. In this case the we model the target as a **linear function** of a set of predictors. Let's call $y$ the target and $X$ a matrix of predictors. $y$ is a 1D column array of $N$ elements, each of which corresponding to one observation. $X$ is a matrix with $N$ rows and $p$ columns. Each row is an observation from a set of $p$ predictors. The form of our model will be $$\hat{y} = X \beta$$ where $\hat{y}$ is our **prediction** of the target $y$ and $\beta$ is a column vector of $p$ elements containing the coefficients of the linear model. The value of the element $\beta_j$ tells us the predictive power that the $j^{\th}$ predictor has on the target. Once we have a model, if we obtain a new observation $x_{\rm new}$ (a row vector of $p$ elements), we can make a prediction for this new observation which will be equal to $$\hat{y}_{\rm new} = x_{\rm new}^{\top} \beta = x_{{\rm new}_1} \beta_1 + x_{{\rm new}_2} \beta_2 + \cdots + x_{{\rm new}_p} \beta_p$$

A probabilistic version of the linear model would attribute the differences between the targets $y$ and the predictions $\hat{y}$ to noise. The most common model assumes that this noise is Gaussian and independent for each observation. Thus $$y-\hat{y} \sim \sigma N(0,1)$$ where $\sigma^2$ is the variance of the noise and $N(0,1)$ is a standard zero-mean Gaussian variable. This implies $$y = X\beta + \sigma {\mathbf n} = N(X\beta,\sigma^2 {\mathbf I})$$ This implies that, in the linear regression model, the $N$ responses are a set of $N$ independent Gaussian variables each with mean $X\beta$ and variance $\sigma^2$.

## Logistic regression for binary responses

If we want to predict a binary response, then the most common generalization of linear regression is called **logistic regression**. Examples of binary responses might be binary choices in a decision making experiment with two alternatives (a two-alternative forced choice experiment), or the presence/absence of a disease, for instance. 

The logistic regression model is automatically probabilistic. Instead of assuming that a linear combination of the predictors is the mean of a set of independent Gaussian variables, as in linear regression, we assume that a linear combination of the predictors **determines** the mean of a set of independent **Bernoulli** random variables, i.e., the probability that each variable (which we can think of as a coin) will land heads. 

Because probabilities have to be positive, but the linear combination can be positive and negative, we need to pass the linear combination through a non-linear function first to obtain something positive. This non-linear function is a **logistic** function in the case of logistic regression. Let's call the linear combination $\eta$. Then, the logistic function is $$p(\eta) = \frac{1}{1+\exp(-\eta)}$$ The inverse of this function, i.e., the function that gives us the linear combination in terms of the mean of the random variables is called a **link** function, and in the case of logistic regression this is the **logit** function, i.e., $$\eta = \log \left( \frac{p}{1-p} \right)$$ 

The general class of models where a linear combination of the predictors specifies, through the use of a link function, the mean of the distribution of the response variable are called **generalized linear models** or **GLMs**. To specify a GLM we need to specify a link function and a probability distribution. A logistic regression model is a particular case of a GLM which uses a logit link function and a Bernoulli probability distribution.

## Multiple regression: Disambiguating true from spurious correlations

Perhaps the most important point in this tutorial is to understand the importance and implications of using **multiple regression**, i.e., of trying to make predictions using multiple simultaneous predictors. This point applies equally to linear and logistic regression, so we will cover it using linear regression which affords some extra geometrical intuitions.

The really important thing to understand is that if we look at the relationship between the responses and each predictor individually, we might be fooled. Some predictors might be strongly associated (i.e., display strong correlations) with the response **only** because they are strongly associated with **other predictors**, even if they are completely independent of the response.

Let's study this phenomenon in a linear regression problem between an output y and two regressors x1 and x2 that have been previously generated. The following code reads the three variables and plots the response as a faunction of each of the two predictors:

In [ ]:
#@title
# read the dataset and plot y against each predictor

df_lr = pd.read_csv('https://raw.githubusercontent.com/arenart73/regression-methods/main/lr_data.csv')
x1  = df_lr['x1'].values  # .values extracts the column as a plain numpy array
x2  = df_lr['x2'].values
y   = df_lr['y'].values
k_y = 1.0   # true coefficient, used in later plots

# create a figure with 2 side-by-side panels sharing the same y axis
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

# zip() pairs each axis with the corresponding predictor array and its label,
# so we can loop over both panels with a single for loop
for ax, x, label in zip(axes, [x1, x2], ['x1', 'x2']):
    ax.scatter(x, y, alpha=0.3, s=10)  # alpha=transparency, s=dot size
    ax.set_xlabel(label, fontsize=13)
    ax.set_ylabel('y', fontsize=13)
    r = np.corrcoef(x, y)[0, 1]  # corrcoef returns a 2×2 matrix; [0,1] picks the off-diagonal (the correlation)
    ax.set_title(f'r = {r:.2f}', fontsize=13)

fig.suptitle('y vs each predictor', fontsize=14)
plt.tight_layout()
plt.show()


display(Markdown(r"""As you can see, both x1 and x2 are clearly correlated with the response variable y"""))
display(Markdown(r"""
Now let us run a univariate linear regression of y on each of the two predictors separately.
In effect, we are fitting the constants $\alpha$ and $\beta$ in the model $y = \alpha + \beta x$.
Although the intercept $\alpha$ should be small, we are letting it not be zero so that it's small
value does not contaminate the estimate of the slope $\beta$.
"""))
display(Markdown(r"""Think about what you expect these coeffients to be before you run the code below"""))

In [ ]:
#@title
# run univariate regressions of y on x1 and x2 separately

import statsmodels.api as sm

# pd.Series wraps the array and gives it a name so statsmodels labels the coefficient correctly
# sm.add_constant prepends a column of 1s so the model fits an intercept as well as a slope
X1  = sm.add_constant(pd.Series(x1, name='x1'))
X2  = sm.add_constant(pd.Series(x2, name='x2'))

# sm.OLS sets up the least-squares problem; .fit() solves it and returns the fitted model
model_x1 = sm.OLS(y, X1).fit()
model_x2 = sm.OLS(y, X2).fit()

print("=== Univariate regression: y ~ x1 ===")
print(model_x1.summary2().tables[1])  # tables[1] is the coefficients table
print()
print("=== Univariate regression: y ~ x2 ===")
print(model_x2.summary2().tables[1])
print()
display(Markdown(r"""As expected, the regression coefficient is different from zero and highly 
significant for both predictors"""))
display(Markdown(r"""
Now let us run a MULTIPLE regression of y on both $x_1$ and $x_2$ at the same time.
The following code creates the matrix $X$ of predictors and then uses the OLS function from statsmodels
to fit the model $y \sim X \beta$.
"""))

In [ ]:
#@title
# run multiple regression of y on x1 and x2 together

# double brackets [['x1','x2']] select multiple columns as a dataframe;
# single brackets ['x1'] would return just one column as a Series
X12 = sm.add_constant(df_lr[['x1', 'x2']])

model_x1x2 = sm.OLS(y, X12).fit()

print("=== Multiple regression: y ~ x1 + x2 ===")
print(model_x1x2.summary2().tables[1])
print()
display(Markdown(r"""
Now Look! The regression coefficient for $x_1$ is still different from zero and highly significant,
but the regression coefficient for $x_2$ is not different from zero and not significant at all!"""))
display(Markdown(r"""
We can also plot the fitted coefficients for the univariate and the multiple regresssions. This is
done in the next cell."""))

In [ ]:
#@title
# plot the coefficients from univariate and multiple regressions so we can compare

# each entry in the dict maps a bar label to the (fitted model, variable name) it should display
models = {
    'y ~ x1\n(univariate)':     (model_x1,   'x1'),
    'y ~ x2\n(univariate)':     (model_x2,   'x2'),
    'y ~ x1 + x2\n(multiple)':  (model_x1x2, 'x1'),
    'y ~ x1 + x2\n(multiple) ': (model_x1x2, 'x2'),
}

labels, coefs, cis, colors = [], [], [], []
palette = {'x1': '#2196F3', 'x2': '#FF5722'}

# loop over the dict; each iteration unpacks the label and the (model, variable) tuple
for label, (model, var) in models.items():
    ci   = model.conf_int().loc[var]  # 95% CI: a row with [lower_bound, upper_bound]
    coef = model.params[var]          # the fitted coefficient for this variable
    labels.append(label)
    coefs.append(coef)
    # convert [lower, upper] to [left_error, right_error] format expected by barh
    cis.append([coef - ci[0], ci[1] - coef])
    colors.append(palette[var])

fig, ax = plt.subplots(figsize=(8, 4))
x_pos = range(len(labels))
# xerr adds the error bars; np.array(cis).T transposes from shape (n,2) to (2,n) as barh requires
ax.barh(x_pos, coefs, xerr=np.array(cis).T, align='center',
        color=colors, alpha=0.7, capsize=5, height=0.5)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(list(x_pos))
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel('Coefficient estimate (± 95% CI)', fontsize=12)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=palette['x1'], label='x1 coefficient'),
                   Patch(color=palette['x2'], label='x2 coefficient')],
          fontsize=10)

plt.tight_layout()
plt.show()

print()
display(Markdown(r"""
As you can see, the coefficient for $x_2$ is large when we run a univariate regresssion, meaning
when we assess only how much $x_2$ is associated to the target, but it's zero when we look at how well
**both** $x_1$ and $x_2$ together predict the target."""))
display(Markdown(r"""**What is going on??**"""))
display(Markdown(r"""Perhaps $x_2$ becomes associated to the target by virtue of being associated with $x_1$...
Let us check if this is the case by plotting $x_1$ vs $x_2$."""))

In [ ]:
#@title
# plot x1 vs x2 to see whether they are correlated with each other

fig, ax = plt.subplots(figsize=(5, 5))

ax.scatter(x1, x2, alpha=0.3, s=10)
ax.set_xlabel('x1', fontsize=13)
ax.set_ylabel('x2', fontsize=13)

r = np.corrcoef(x1, x2)[0, 1]  # Pearson correlation; [0,1] picks the off-diagonal element
ax.set_title(f'r = {r:.2f}', fontsize=13)

fig.suptitle('x2 vs x1', fontsize=14)
plt.tight_layout()
plt.show()

print()
display(Markdown(r"""Indeed, we can see that $x_1$ and $x_2$ are strongly correlated. The picture that emerges
is that $x_1$ is trully associated with the target $y$ and **also** with $x_1$. As we have seen
multiple regression can help us with exactly this kind of situation."""))

In [ ]:
#@title
# explanation of the multiple regression coefficient as a residual regression

display(Markdown(r"""In fact, one can show that..."""))
display(Markdown(r""" ... **the multiple regression coefficient of $y$ on a predictor $x$
is equal to the univariate regression coefficient of $y$ on the RESIDUAL of $x$ after trying to linearly
predict $x$ using all other available predictors.**"""))
display(Markdown(r"""If all the predictive power of $x$ about the target
comes about from it's linear association with other predictors, then the residual of regressing $x$
with the remaining predictors will not be associated to the target $y$ at all, and thus the multiple
regression coefficient for $x$ will be zero."""))
display(Markdown(r"""The next cell demonstrates this by computing the residuals of $x_1$ and $x_2$ first,
and then performing univariate regressions of $y$ on these residuals."""))

In [ ]:
#@title
# compute residuals of x1 and x2 after regressing each on the other,
# then run univariate regressions of y on these residuals

# regress x1 on x2, take the residuals = the part of x1 not explained by x2
# pd.Series(..., name='x1') wraps the result so statsmodels labels the coefficient correctly
x1_resid = pd.Series(sm.OLS(x1, sm.add_constant(x2)).fit().resid, name='x1')
x2_resid = pd.Series(sm.OLS(x2, sm.add_constant(x1)).fit().resid, name='x2')

# confirm orthogonality: correlation between x2 and x1_resid should be ~0
print(f"Correlation x2 vs x1_resid: {np.corrcoef(x2, x1_resid)[0,1]:.4f}  (should be ~0)")
print(f"Correlation x1 vs x2_resid: {np.corrcoef(x1, x2_resid)[0,1]:.4f}  (should be ~0)")

# univariate regressions on the orthogonalized predictors
model_x1_orth = sm.OLS(df_lr['y'], sm.add_constant(x1_resid)).fit()
model_x2_orth = sm.OLS(df_lr['y'], sm.add_constant(x2_resid)).fit()

print("\n=== y ~ x1_resid (should match x1 coef in multiple regression) ===")
print(model_x1_orth.summary2().tables[1])
print("\n=== y ~ x2_resid (should match x2 coef in multiple regression) ===")
print(model_x2_orth.summary2().tables[1])

print()
display(Markdown(r"""As you can see, now the univariate regression of $y$ on the residual of $x_2$
matches the multiple regression coefficient for $x_2$, and is zero."""))
display(Markdown(r"""Let us plot the coefficients from all the different regresssions so we can compare:"""))

In [ ]:
#@title
# plot coefficients from all six regressions: univariate, multiple, and FWL residual regressions

# same dict pattern as before: label → (fitted model, variable name)
models_fwl = {
    'y ~ x1\n(univariate)':      (model_x1,      'x1'),
    'y ~ x2\n(univariate)':      (model_x2,      'x2'),
    'y ~ x1 + x2\n(multiple)':   (model_x1x2,    'x1'),
    'y ~ x1 + x2\n(multiple) ':  (model_x1x2,    'x2'),
    'y ~ x1⊥x2\n(FWL)':          (model_x1_orth, 'x1'),
    'y ~ x2⊥x1\n(FWL)':          (model_x2_orth, 'x2'),
}

labels, coefs, cis, colors = [], [], [], []
palette = {'x1': '#2196F3', 'x2': '#FF5722'}

for label, (model, var) in models_fwl.items():
    ci   = model.conf_int().loc[var]  # 95% CI row for this variable
    coef = model.params[var]          # fitted coefficient
    labels.append(label)
    coefs.append(coef)
    cis.append([coef - ci[0], ci[1] - coef])  # convert to [left_error, right_error]
    colors.append(palette[var])

fig, ax = plt.subplots(figsize=(8, 5))
x_pos = range(len(labels))
ax.barh(x_pos, coefs, xerr=np.array(cis).T, align='center',
        color=colors, alpha=0.7, capsize=5, height=0.5)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(k_y, color='gray', linewidth=0.8, linestyle=':')  # vertical line at the true value
ax.set_yticks(list(x_pos))
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel('Coefficient estimate (± 95% CI)', fontsize=12)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=palette['x1'], label='x1 coefficient'),
                   Patch(color=palette['x2'], label='x2 coefficient'),
                   plt.Line2D([0], [0], color='gray', linestyle=':', label=f'true k_y = {k_y}')],
          fontsize=10)

plt.tight_layout()
plt.show()

print()
display(Markdown(r"""The plot confirms that the coefficients of the univariate regressions on the
residuals (but NOT of the actual data!!) are equal to the coefficients of the multivariate regression."""))
print()
display(Markdown(r"""Analyzing the geometrical relationship between the predictors and the target
is very useful to understand the problem we have been studying."""))
display(Markdown(r"""In the next cell, we make two kinds of plots of x1 x2 and y."""))

In [ ]:
#@title
# two kinds of plots of x1, x2, y

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 6))

# --- Left: standard regression view — data scatter ---
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(x1, x2, y, alpha=0.2, s=5, color='steelblue')
ax1.set_xlabel('x1', fontsize=11)
ax1.set_ylabel('x2', fontsize=11)
ax1.set_zlabel('y', fontsize=11)
ax1.set_title('Data: y as a function of x1 and x2', fontsize=11)

# --- Right: geometric view — data vectors in "observation space" ---
ax2 = fig.add_subplot(122, projection='3d')

theta        = np.radians(15)   # convert 15 degrees to radians (required by sin/cos)
x1_vec       = np.array([0, 1, 0])                                    # x1 points along the y-axis
y_vec        = np.array([0, np.cos(theta), np.sin(theta)])             # y is in the yz-plane, 15° from x1
x2_vec       = np.array([0.8 * np.sin(theta), 0.8 * np.cos(theta), 0])# x2 is in the xy-plane, 15° from x1
x2_proj_vec  = np.dot(x2_vec, x1_vec) * x1_vec  # projection of x2 onto x1: (x2·x1) * x1 (since |x1|=1)
x2_resid_vec = x2_vec - x2_proj_vec              # residual: what's left of x2 after projecting onto x1

print(f"x2  · y   = {np.dot(x2_vec,       y_vec):.3f}  (non-zero)")
print(f"x2⊥ · y   = {np.dot(x2_resid_vec, y_vec):.3f}  (zero → perpendicular)")

# helper function: draw a 3D arrow from the origin to point v
def quiv(ax, v, color, label):
    ax.quiver(0, 0, 0, v[0], v[1], v[2], color=color,
              arrow_length_ratio=0.12, linewidth=2.5)
    ax.text(*(v * 1.15), label, color=color, fontsize=11, fontweight='bold')  # label placed beyond tip

quiv(ax2, x1_vec,       '#2196F3', 'x1')
quiv(ax2, y_vec,        '#4CAF50', 'y')
quiv(ax2, x2_vec,       '#FF9800', 'x2')
quiv(ax2, x2_proj_vec,  '#9E9E9E', 'proj(x2→x1)')
quiv(ax2, x2_resid_vec, '#E91E63', 'x2⊥x1')

# dashed lines showing that x2 = proj(x2→x1) + x2_resid
for start, end, c in [
    (x2_proj_vec,  x2_vec, '#E91E63'),
    (x2_resid_vec, x2_vec, '#9E9E9E'),
]:
    ax2.plot([start[0], end[0]], [start[1], end[1]], [start[2], end[2]],
             color=c, linestyle='--', linewidth=1, alpha=0.5)

ax2.set_xlabel('X', fontsize=11)
ax2.set_ylabel('Y', fontsize=11)
ax2.set_zlabel('Z', fontsize=11)
ax2.set_xlim([0, 0.8]); ax2.set_ylim([0, 1.1]); ax2.set_zlim([0, 0.6])
ax2.set_title('x2⊥x1 lies on the x-axis → perpendicular to y in the yz-plane', fontsize=10)

plt.tight_layout()
plt.show()

print()
display(Markdown(r"""The plot on the left is the standard way one thinks about this problem. This plot
shows a scatter of all the ($x_1$, $x_2$, $y$) triplets. Because the variables are linearly related, the
data forms a plane in 3D."""))
display(Markdown(r"""The plot on the right is the really interesting and important one. It is a plot of
$x_1$, $x_2$ and $y$ in 'data space'. If you remember, each of these arrays have a dimension equal to the
number of observations (in our case $N=1000$), so the plot would really have to be in 1000 dimensions.
Since we unfortunately cannot make plots in 1000 dimensions, we have made a 3D plot (as if the dataset
only contained 3 observations). The key here is the geometrical relationship between the $x_1$, $x_2$ and 
$y$ vectors, and the geometrical form of the residual of $x_2$ on $x_1$, which we denote 
by $x_2 \perp x_1$."""))
display(Markdown(r"""Think carefully about this plot keeping in mind that the univariate regression 
coefficient between two arrays is proportional to their dot product: 
$\beta = \langle x, y \rangle/\langle x,x \rangle$."""))
display(Markdown(r"""Do you see why this is a good representation of the data in our problem?"""))
display(Markdown(r"""Can you guess where the predicted estimate $\hat{y} = X \beta$ would be on this plot?"""))

# Generalized Linear Models: Logistic Regression

Remember, the difference with linear regression is that now the response or target variable $y$ is binary. The linear combination of the regressors is passed through a logistic function to obtain the probability that the target is +1 (a coin flip). Let's make a plot of how this looks for a single regressor $x$.

In [ ]:
#@title
# plot an example of a logistic regression: regressor, probability curve, and binary responses

rng_ex = np.random.default_rng(seed=1)  # fixed seed so the plot is reproducible

beta_ex = 2.0
n_ex    = 100

x_ex = rng_ex.normal(0, 1, n_ex)               # 100 random stimulus values from a Gaussian
p_ex = 1 / (1 + np.exp(-beta_ex * x_ex))       # logistic function: maps linear predictor to probability
r_ex = rng_ex.binomial(1, p_ex)                 # draw a 0/1 response for each trial using its probability

# 300 evenly spaced points for a smooth sigmoid curve
x_line = np.linspace(x_ex.min() - 0.5, x_ex.max() + 0.5, 300)
p_line = 1 / (1 + np.exp(-beta_ex * x_line))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_line, p_line, color='steelblue', linewidth=2, label='logistic curve')

# plot responses at y=1 and y=0; facecolors='none' gives open circles
ax.scatter(x_ex[r_ex == 1], np.ones(r_ex.sum()),
           facecolors='none', edgecolors='#4CAF50', s=60, linewidths=1.5, label='r = 1')
ax.scatter(x_ex[r_ex == 0], np.zeros((r_ex == 0).sum()),
           facecolors='none', edgecolors='#E91E63', s=60, linewidths=1.5, label='r = 0')
ax.set_xlabel('x', fontsize=13)
ax.set_ylabel('p(r = 1 | x)', fontsize=13)
ax.set_title(f'Logistic regression example  (β = {beta_ex})', fontsize=13)
ax.set_ylim(-0.15, 1.15)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print()
display(Markdown(r"""When the regressor is positive, the output tends to be +1, and when it is 
negative, the output tends to be zero. But there are exceptions where, for instance, the input is
positive but the output is zero. If we were using logistic regression to model binary decisions, 
those cases would be errors."""))
display(Markdown(r"""The probability of an error (or of a correct choice) can be calculated from the
blue logistic curve."""))
print()
display(Markdown(r"""**UNDERSTANDING BINARY CHOICE EXPERIMENTS USING LOGISTIC REGRESSION**"""))
display(Markdown(r"""Now we are going to use logistic regression to understand multiple factors that 
can influence the choices of subjects on sensory discrimination tasks. As we mentioned previously,
in addition to the sensory stimulus, a common additional influence is choices made in previous trials.
In particular subjects tend to **repeat choices**."""))
display(Markdown(r"""We will explore this phenomenon using a previously generated artificial dataset
which emulates 5000 trials from a binary choice experiment. The next cell loads this dataset, which 
consists of 3 column vectors cotaining the choice, stimulus and previous choice for each trial in 
the simulated experiment."""))

In [ ]:
#@title
# load data for exploring binary choice

df_task = pd.read_csv('https://raw.githubusercontent.com/arenart73/regression-methods/main/task_data.csv')
beta_x    = 2.0   # true stimulus weight, used in later cells
beta_prev = 0.5   # true previous-choice weight, used in later cells

display(Markdown(r"""**CONDITIONAL PSYCHOMETRIC FUNCTIONS**"""))
display(Markdown(r"""Given this (artificial) dataset, a first "common sense" approach to test if choices 
on a particular trial are affected by choices in previous trials, is to calculate a **conditional psychometric** 
function."""))
display(Markdown(r"""The psychometric function is a plot of the probability of making one of the two possible 
choices as a function of the strength of evidence of the stimulus. It has a sigmoidal function (do you 
understand why?)."""))
display(Markdown(r"""A conditional psychometric is the same thing but using only trials to meet some condition,
in this case, that the choice was +1 (or -1) in the previous trial.""")) 
display(Markdown(r"""The next cell computes the conditional psychometrics from our dataset and plots them."""))


In [ ]:
#@title
# compute and plot psychometric functions conditioned on previous choice

# recode choice from +1/-1 to 1/0 so that .mean() gives the proportion of +1 choices
df_task['choice_01'] = (df_task['choice'] == 1).astype(float)

fig, ax = plt.subplots(figsize=(8, 4))

for prev, color, label in [(1,  '#2196F3', 'prev choice = +1 (right)'),
                            (-1, '#E91E63', 'prev choice = −1 (left)')]:
    # select only trials where prev_choice equals the current value of prev
    df_cond = df_task[df_task['prev_choice'] == prev]
    # groupby groups trials by stimulus level; .mean() gives the proportion of +1 choices in each group
    psy = df_cond.groupby('x')['choice_01'].mean()
    ax.scatter(psy.index, psy.values, color=color, label=label, s=50)  # index=x levels, values=proportions

ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
ax.axvline(0,   color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('Stimulus (x)', fontsize=13)
ax.set_ylabel('p(choice = +1 | x)', fontsize=13)
ax.set_title('Psychometric function conditioned on previous choice', fontsize=13)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print()
display(Markdown(r"""Because the two conditional psychometric functions are different, choices about 
the stimulus seem to be affected by previous choice in this (artificial) dataset. A logistic regression 
model can help us quantify the relative contribution of the stimulus and previous choice to the current 
choice."""))
display(Markdown(r"""The next cell fits a logistic regression model to the data."""))

In [ ]:
#@title
# fit a logistic regression model predicting choice from stimulus and previous choice

# dropna() removes the first trial (NaN prev_choice); .copy() prevents modifying the original dataframe
df_fit = df_task.dropna().copy()
# recode choice from +1/-1 to 1/0: (+1+1)/2 = 1, (-1+1)/2 = 0 — required by sm.Logit
df_fit['choice_01'] = (df_fit['choice'] + 1) / 2

X_task     = sm.add_constant(df_fit[['x', 'prev_choice']])  # predictor matrix with intercept
model_task = sm.Logit(df_fit['choice_01'], X_task).fit()    # fit logistic regression

print(model_task.summary2().tables[1])
display(Markdown(r"""We see that the coefficient associated to the stimulus is approxiately 2 and the 
coeffienct associated to the previous choice of the subject is approximately 0.6. How should we interpret
these numbers?"""))

# Interpreting of the coefficients of a logistic regression model

Consider a simple one variable model first. In this case, because we assume that the probability of choosing one option is a logistic function of the linear combination of variables (let's denote by $p_+$ the probability of choosing +1), it is easy to show that $$\log\left(\frac{p_+}{1 -p_+}\right) = \beta_0 + \beta_1 x$$

The quantity inside the logarithm is called the **odds** of choosing +1, i.e., how much more likely it is to choose +1 than the remaining options. Thus, the previous formula means that the change in odds when the regressor $x$ changes by 1 unit is $$\Delta {\rm Odds} = \exp (\beta_1) $$

If the regressor $x$ is a categorical binary variable coded as $\{ 0,1 \}$, then the corresponding coefficient $\beta_1$ measures the change in odds for each category. For instance, if the binary response variable is being dead after 5 years, and the categorical regressor is being a smoker for the last 5 years, then if the fit of the logistic regression model produces $$\Delta {\rm Odds} = 3$$ this means that it's 3 times more likely to be dead after 5 years if one was a smoker than if one wasn't (note that if the categorical regressor is coded as $\{ -1, +1 \}$ then the relationship between $\beta_1$ and the Odds changes accordingly).

If there are multiple regressors, this interpretation holds as long as, when we change a regressor of interest, all other regressors are held constant. Note that, in the absence of interaction terms, the definition of the multivariate logistic regression model implies that the change in log-Odds from changing one variable is the same regardless of the values of the other variables.

If there are interaction terms, for instance, if the model is $$\log\left(\frac{p_+}{1 -p_+}\right) = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \beta_{\rm int} x_1 x_2$$ then the change in odds from changing $x_1$ by 1 unit will depend both on the value of $x_2$ and on the value of the interaction coefficient $\beta_{\rm int}$. 

Because the interpretation of the coefficients depends critically on the units with which we measure the regressors, we typically standardize continuous regressors before fitting a logistic (or linear) regression model. Standardizing categorical regressors (coded as $\{ 0, 1 \}$ or $\{ -1, +1 \}$) is not generally needed. 



# Work problem: How do mice react to errors during sensory discrimination?

Now we will work on an example based on work from our lab describing how mice react to errors in sensory discrimination. In this experiment, mice have to respond by licking from a spout on the left or the right in front of them depending on whether the frequency of a pure tone presented to them is higher or lower than a particular reference tone, which they learn by trial and error.

After learning the task, mice perform 200-300 trials during one behavioral session lasting approximately 1 h. Because we make the frequency of the tones close enough to the reference, this auditory discrimination task is difficult, and mice make mistakes approximately 20-25% of the trials. 

The question is: **how does an error in one trial affect the behavior of the mice in the next trial?**

To study this problem, we created an artificial dataset of 5000 trials that has the same properties that we observe in the behavioral experiments. The next cell loads this dataset.

In [ ]:
#@title
# load data to study effect of errors on subsequent choices

df_task2 = pd.read_csv('https://raw.githubusercontent.com/arenart73/regression-methods/main/task2_data.csv')

# true parameter values, used in later cells
beta_x2       =  2.0
beta_prev2    =  0.5
beta_out2     =  0.0
beta_xout2    =  0.4
beta_prevout2 = -0.6

print("Dataset df_task2 ready.")
# print(f"Trials: {len(df_task2)}  |  columns: {list(df_task2.columns)}")
display(Markdown(r"""To explore whether errors make a difference to subsequent choices, let us first
calculate the accuracy of choices after a correct trial and after an error trial."""))

In [ ]:
#@title
# compute accuracy after correct trials and after error trials

df2 = df_task2.dropna().copy()   # drop first trial (NaN), make a copy
df2 = df2[df2['x'] != 0]        # exclude x=0 trials — no unambiguous correct answer

# a trial is correct if the sign of the choice matches the sign of the stimulus
# & is element-wise AND for boolean arrays; | is element-wise OR; \ continues on the next line
df2['correct'] = ((df2['x'] > 0) & (df2['choice'] == 1)) | \
                 ((df2['x'] < 0) & (df2['choice'] == -1))

# compute accuracy separately for trials after a correct (prev_outcome=-1) and after an error (prev_outcome=+1)
for prev_o, label in [(-1, 'After correct'), (1, 'After error')]:
    subset = df2[df2['prev_outcome'] == prev_o]
    acc = subset['correct'].mean()  # proportion of correct trials
    n   = len(subset)
    print(f"{label:15s}  accuracy = {acc:.3f}  (n = {n})")

print()
display(Markdown(r"""As you can see, accuracy after errors is almost 10% higher than after correct trials.
This suggests that both the strength of evidence of the stimulus and the previous outcome of the trial 
contribute to the accuracy of each given choice. How can we assess the relative contribution of each of 
these two influences?"""))
display(Markdown(r"""**Yes!!** using logistic regression. What would be the most direct way to do it?"""))
display(Markdown(r"""Think about it for little while..."""))

In [ ]:
#@title
# a strategy for measuring post-error increases in accuracy

display(Markdown(r"""The most direct way to do it is to run a logistic regresssion but one in which
instead of predicting choice, we predict whether the trial was correct or an error."""))
display(Markdown(r"""For this the regressors would be the absolute value of the stimulus (why?) and
the outcome (correct/error) of the previous trial."""))
display(Markdown(r"""The next cell fits this model and plots the coefficients."""))

In [ ]:
#@title
# fit a logistic regression predicting trial accuracy from |stimulus| and previous outcome

df2['abs_x']      = df2['x'].abs()       # |x|: larger values = stronger evidence regardless of direction
df2['correct_01'] = df2['correct'].astype(int)  # convert True/False to 1/0

X_out     = sm.add_constant(df2[['abs_x', 'prev_outcome']])
model_out = sm.Logit(df2['correct_01'], X_out).fit()

print(model_out.summary2().tables[1])

# plot the two coefficients (intercept excluded)
predictors = ['abs_x', 'prev_outcome']
coefs = model_out.params[predictors]          # fitted coefficients for these two predictors
cis   = model_out.conf_int().loc[predictors]  # confidence intervals: dataframe with [lower, upper] columns

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(predictors, coefs,
        xerr=[coefs - cis[0], cis[1] - coefs],  # convert [lower,upper] to [left_err, right_err]
        color=['#2196F3', '#FF9800'], alpha=0.7, capsize=5, height=0.4)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Coefficient (log-odds)', fontsize=12)
ax.set_title('Logistic regression: predicting correct outcome', fontsize=12)
plt.tight_layout()
plt.show()

display(Markdown(r"""Ok, so the log-odds from the stimulus is about 3 times as large as that from the 
previous outcome. This makes sense, the previous outcome modulates the effect of the stimulus."""))

display(Markdown(r"""Now the big question is: **What could be the possible sources through which errors make 
a subject better at discrimiating the stimulus?**"""))

display(Markdown(r"""Think about this for a few minutes and think also how you would test your hypothesis 
using logistic regression"""))

In [ ]:
#@title
# sources of post error increase in accuracy

display(Markdown(r"""'The most obvious source through which errors might affect accuracy is to draw more 
attention to the sensory stimulus in the next trial. How could we test if this is the case using Logistic 
regression?"""))

display(Markdown(r"""If the subject pays more attention to the stimulus, then it is as if the stimulus 
was STRONGER. We can model this using an interaction term between the stimulus and the previous outcome,
which would capture an effective increase in the stimulus weight after errors."""))

display(Markdown(r"""So now we will run a logistic regression model trying to predict choice (not accuracy!) 
using as regressors the stimulus, the previous outcome, and their interaction. Let us see what happens..."""))


In [ ]:
#@title
# fit logistic regression predicting choice from stimulus, previous outcome, and their interaction

df3 = df_task2.dropna().copy()
df3['choice_01']        = (df3['choice'] == 1).astype(int)
# the interaction term is the element-wise product of the two columns on each trial
df3['x_x_prev_outcome'] = df3['x'] * df3['prev_outcome']

X3     = sm.add_constant(df3[['x', 'prev_outcome', 'x_x_prev_outcome']])
model3 = sm.Logit(df3['choice_01'], X3).fit()

print(model3.summary2().tables[1])

predictors = ['x', 'prev_outcome', 'x_x_prev_outcome']
coefs = model3.params[predictors]
cis   = model3.conf_int().loc[predictors]

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(predictors, coefs,
        xerr=[coefs - cis[0], cis[1] - coefs],
        color=['#2196F3', '#FF9800', '#9C27B0'], alpha=0.7, capsize=5, height=0.4)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Coefficient (log-odds)', fontsize=12)
ax.set_title('Logistic regression: predicting choice\n(stimulus, prev outcome, interaction)', fontsize=11)
plt.tight_layout()
plt.show()

display(Markdown(r"""Note that, unlike our previous model where we predicted accuracy, here the coefficient 
for the previous outcome is not significant. This is because the previous outcome (correct or error)
does not induce a tendency for the subject to choose one side or the other. Instead, what it does
is to modulate the influence of the stimulus on choice. Since we are coding errors as +1
an improvement in accuracy after errors leads to a positive magnitude for the interaction coefficient."""))

display(Markdown(r"""It is important to always keep this distinction in mind: main effects capture 
direct influences of the regressor on the predicted variable, i.e., a tendency for changes in the regressor 
to be associated with changes in the value of the predicted variable. But the fact that there is no main 
effect does not mean that a predictor has no effect on the predicted variable. Such effect might exist, but
only reveal itself as a tendency to alter the influence of another predictor."""))

In [ ]:
#@title
# another source of post error increases in accuracy

display(Markdown(r"""There is another, more subtle, way in which previous outcome might affect choice..."""))

display(Markdown(r"""This is through an effect of previous outcome on the tendency to repeat choices that we studied above."""))

display(Markdown(r"""Let us unpack this: First of all, we should check if there is a tendency to repeach choices 
in this dataset. To know whether this is the case, we can compute the conditional psychometrics separately after each
of the two possible choices. Let us do it, and plot them together with the unconditional psychometric:"""))

In [ ]:
#@title
# plot psychometric functions conditioned on previous choice, for the task2 dataset

df_psy2 = df_task2.dropna().copy()
# recode choice from +1/-1 to 1/0 so the mean gives the proportion of +1 choices
df_psy2['choice_01'] = (df_psy2['choice'] == 1).astype(float)

fig, ax = plt.subplots(figsize=(5, 5))

# unconditional psychometric: all trials pooled
psy_all = df_psy2.groupby('x')['choice_01'].mean()
ax.scatter(psy_all.index, psy_all.values, color='black', label='unconditional', s=50)

# conditional psychometrics: one curve per previous choice value
for prev, color, label in [(1,  '#2196F3', 'prev choice = +1 (right)'),
                            (-1, '#E91E63', 'prev choice = −1 (left)')]:
    df_cond = df_psy2[df_psy2['prev_choice'] == prev]  # keep only trials with this previous choice
    psy = df_cond.groupby('x')['choice_01'].mean()     # proportion of +1 choices at each stimulus level
    ax.scatter(psy.index, psy.values, color=color, label=label, s=50)

ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
ax.axvline(0,   color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('Stimulus (x)', fontsize=13)
ax.set_ylabel('p(choice = +1 | x)', fontsize=13)
ax.set_title('Psychometric function conditioned on previous choice', fontsize=13)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

display(Markdown(r"""From this figure, it should be clear to you that the tendency to repeat previous choices 
has the effect of lowering accuracy. The black curve is approximately the average of the two colored curves,
and thus it will have a slightly lower maximal slope than that of the conditional psychometrics."""))

display(Markdown(r"""Because whether a choice is correct only depends on the stimulus in the current trial, 
any influence on choice by anything other than the stimulus can only reduce accuracy."""))

display(Markdown(r"""So... if a tendency to repeat choices lowers accuracy, perhaps errors can tend to 
increase accuracy by reducing the tendency to repeat previous choices?"""))

display(Markdown(r"""How could we test this using Logistic regression? Using another interaction term!
This time, an interaction between previous choice and previous outcome. Because an improvement in
accuracy would come about through a REDUCTION on the tendency to repeat, we would expect the sign
of the coefficient for the interaction term to be negative in this case. Let us test that!"""))

In [ ]:
#@title
# fit the full logistic regression model with all 5 regressors and both interaction terms

df4 = df_task2.dropna().copy()
df4['choice_01']                  = (df4['choice'] == 1).astype(int)
df4['x_x_prev_outcome']           = df4['x']           * df4['prev_outcome']   # stimulus × outcome interaction
df4['prev_choice_x_prev_outcome'] = df4['prev_choice'] * df4['prev_outcome']   # choice × outcome interaction

X4     = sm.add_constant(df4[['x', 'prev_choice', 'prev_outcome',
                               'x_x_prev_outcome', 'prev_choice_x_prev_outcome']])
model4 = sm.Logit(df4['choice_01'], X4).fit()

print(model4.summary2().tables[1])

# dict pairing each regressor name to the true coefficient used when generating the data
true_values = {
    'x':                         beta_x2,
    'prev_choice':                beta_prev2,
    'prev_outcome':               beta_out2,
    'x_x_prev_outcome':           beta_xout2,
    'prev_choice_x_prev_outcome': beta_prevout2,
}
print("\nTrue parameter values:")
for name, val in true_values.items():
    print(f"  {name:30s}  {val}")

# coefficient plot
predictors = ['x', 'prev_choice', 'prev_outcome',
              'x_x_prev_outcome', 'prev_choice_x_prev_outcome']
coefs = model4.params[predictors]
cis   = model4.conf_int().loc[predictors]

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#E91E63']

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(predictors, coefs,
        xerr=[coefs - cis[0], cis[1] - coefs],
        color=colors, alpha=0.7, capsize=5, height=0.5)

# draw a black tick mark at the true value for each regressor
for name, val in true_values.items():
    # predictors.index(name) gives the y-position (bar number) for this regressor
    ax.plot(val, predictors.index(name), 'k|', markersize=12, markeredgewidth=2)

ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Coefficient (log-odds)', fontsize=12)
ax.set_title('Full logistic regression: predicting choice', fontsize=12)
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0], [0], color='k', marker='|', linestyle='none',
                          markersize=10, markeredgewidth=2, label='true value')],
          fontsize=10)
plt.tight_layout()
plt.show()

display(Markdown(r"""So we see that in this dataset, errors led to both an increase in the weight
of the stimulus and also to a decrease in the tendency to repeat the previous choice. Both of these 
factors contribute the the observed increase in accuracy after errors."""))

display(Markdown(r"""In the figure you also see the true value of the coefficients. Indeed, this 
model that we just fit is the model that I used to generate the data. As the number of simulated trials
tends to infinity we would expect to perfectly recover the true values of the coefficients."""))

# Model Comparison: Cross Validation

So far we have fitted linear and logistic regression models to data. But how does one evaluate how accurate the model is? Equivalently, if there are several candidate models that we can fit to the same dataset, how
do we compare these models to decide which one is best?

There are many ways to do it, but we are going to cover a particularly intuitive one: **cross validation**.

Cross validation evaluates models in terms of how well they **generalize**, i.e., how well they can predict data that was not used to fit the model. This requires separating the data into training and testing sets.
**Training data** is the part of the data that we use to train a model, i.e., to learn the parameters of the model, in our case the regression coefficients. **Test data** is a **separate** set of observations that we should **not** use to train the model, and instead we will use to generate model predictions (after the model has been learned). Evaluating the accuracy of these predictions will give us a sense of how accurate the model is. 

## Procedure for K-fold cross validation

To perform cross validation, the whole dataset of $N$ observations is randomly divided into $K$ different subsets, or folds (each containing $N/K$ observations). In each fold, 1 of these subsets is set aside as test data, and the remaining $K-1$ are used to train the model. Once the model is trained, we use the test data for that fold to generate predictions. This process is repeated for the $K$ folds, so that at the end we have $N$ predictions. 

Then we evaluate the accuracy of these predictions. If the responses are continuous, as in linear regression, we can use the MSE (mean squared error) between predictions and actual responses to test accuracy. If the data is binary, as in logistic regression, one can use the AUC (area under the curve) of the logistic classifier as a measure of accuracy.

The value of $K$ is typically 5 or 10, we can discuss in the class how to set this value.

## Model complexity and overfitting

One often wants to compare models with different levels of "complexity". Informally, models with more parameters are more complex and can provide more accurate fits to data. Thus, we expect **training error** (errors predicting the data used to fit the model) should uniformly decrease as the complexity of the model increases. 

However, there is a catch. If the model is too complex, it will start "fitting the noise". This is called **overfitting**. Overfitting takes place when the model learns aspects of the data that are not generic and don't generalize to other datasets created using the same data generating process. Because of this, if a model is overfitting, it will generalize poorly and will make inaccurate predictions on the test data.

Cross validation evaluates performance on test data, and thus can be used to assess overfitting and, consequently, to decide on the right amount of model complexity. Although we will not do this here, models often include a **regularization** parameter that controls the complexity of the model. Cross validation can be used to choose the optimal magnitude of the regularization parameter leading to optimal generalization. 

# Example for logistic regression

We will consider an example of these concepts using logistic regression. The example considers two datasets. One is a dataset very similar to the one we were just working with, which was generated using two interaction terms between previous outcome and the stimulus and previous choice. The other dataset comes from the same model but setting the two interaction terms to zero. 

We will then fit two logistic regression models to the two datasets. One model has three regressors: stimulus, previous choice and previous outcome. The other one has five: the previous three plus the two interaction terms. 

We will use cross validation to evaluate the performance of each model on each dataset, using the AUC.

Think for a few minutes what you expect the results will be when we do this exercise...

When you are ready, you can proceed. The first cell reads the two datasets which were previously generated and defines the matrix with the regressors that each model uses.

In [ ]:
#@title
# reading datasets and making models to compare using cross validation

from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

# -------------------------------------------------------
# Change n to explore the effect of dataset size.
# Try for example: 200, 500, 1000, 5000
# -------------------------------------------------------
n = 200

# Load both datasets.
# dropna() removes the first trial, which has NaN for prev_choice and prev_outcome.
# head(n) keeps only the first n trials.
# reset_index(drop=True) renumbers rows from 0 — needed so the CV loop can
# correctly index into the arrays using iloc (iloc uses row position, not row label).
df_A = pd.read_csv('https://raw.githubusercontent.com/arenart73/regression-methods/main/task3_data.csv').dropna().head(n).reset_index(drop=True)
df_B = pd.read_csv('https://raw.githubusercontent.com/arenart73/regression-methods/main/task3_null_data.csv').dropna().head(n).reset_index(drop=True)

# Add the two interaction columns to both datasets upfront,
# so the full model can use them without extra steps inside the loop.
for df in [df_A, df_B]:
    df['x_x_prev_outcome']           = df['x'] * df['prev_outcome']
    df['prev_choice_x_prev_outcome'] = df['prev_choice'] * df['prev_outcome']

# Recode choice from +1/-1 to 1/0 — required by the logistic regression function.
df_A['choice_01'] = (df_A['choice'] == 1).astype(int)
df_B['choice_01'] = (df_B['choice'] == 1).astype(int)

# Define which columns each model uses as predictors.
simple_predictors = ['x', 'prev_choice', 'prev_outcome']
full_predictors   = ['x', 'prev_choice', 'prev_outcome',
                     'x_x_prev_outcome', 'prev_choice_x_prev_outcome']

print(f"Dataset A (with interactions): {len(df_A)} trials")
print(f"Dataset B (no interactions):   {len(df_B)} trials")

display(Markdown(r"""The next cell fits two models to the two datasets using cross validation, and
then computes the accuracy of the predictions on the teset data using the AUC. It outputs the
AUC for each case and then plots the results as a bar graph."""))

In [ ]:
#@title
# running cross validation on different models/datasets and plotting results

from sklearn.linear_model import LogisticRegression

# We compare 2 models × 2 datasets = 4 combinations.
# For each combination, 5-fold CV gives us n out-of-sample probability predictions,
# which we evaluate using AUC (Area Under the ROC Curve).
# AUC = 0.5 means the model is no better than chance.
# AUC = 1.0 means the model perfectly predicts every trial.

auc_results = {}

datasets = {
    'Dataset A\n(with interactions)': df_A,
    'Dataset B\n(no interactions)':   df_B,
}
models = {
    'Simple model\n(3 regressors)': simple_predictors,
    'Full model\n(5 regressors)':   full_predictors,
}

# Outer loop: iterate over the two datasets.
for dataset_name, df in datasets.items():

    # Inner loop: iterate over the two models.
    for model_name, predictors in models.items():

        # Set up 5-fold cross-validation.
        # shuffle=True randomises the order before splitting.
        # random_state=42 makes the split reproducible.
        kf = KFold(n_splits=5, shuffle=True, random_state=42)

        # Pre-allocate arrays to accumulate predictions across all 5 folds.
        all_probs  = np.zeros(len(df))
        all_labels = np.zeros(len(df))

        # Loop over the 5 folds.
        for train_idx, test_idx in kf.split(df):

            df_train = df.iloc[train_idx]
            df_test  = df.iloc[test_idx]

            # Extract predictor values as plain arrays.
            # sklearn expects arrays rather than dataframes, so we use .values.
            X_train = df_train[predictors].values
            y_train = df_train['choice_01'].values
            X_test  = df_test[predictors].values

            # Fit logistic regression using sklearn, which is more robust than
            # statsmodels inside CV loops (it handles near-singular cases gracefully).
            # C=1e4 sets very weak regularization — large C means almost no penalty,
            # so the result is close to an ordinary unregularized logistic regression.
            # max_iter=1000 gives the optimizer enough iterations to converge.
            model_fit = LogisticRegression(C=1e4, max_iter=1000)
            model_fit.fit(X_train, y_train)

            # predict_proba returns a 2-column array: [P(choice=0), P(choice=1)].
            # We take the second column [:,1] = probability that choice = +1.
            # These are out-of-sample predictions: the model never saw these trials.
            all_probs[test_idx]  = model_fit.predict_proba(X_test)[:, 1]
            all_labels[test_idx] = df_test['choice_01'].values

        # Compute AUC from the full set of out-of-sample predictions.
        auc = roc_auc_score(all_labels, all_probs)
        auc_results[(dataset_name, model_name)] = auc

        label_a = dataset_name.replace('\n', ' ')
        label_b = model_name.replace('\n', ' ')
        print(f"{label_a:35s}  |  {label_b:25s}  |  AUC = {auc:.3f}")

# plotting results

# Plot the 4 AUC values as a grouped bar chart.
# Each group of bars corresponds to one dataset;
# within each group, the two bars correspond to the two models.

dataset_names = list(datasets.keys())
model_names   = list(models.keys())

# Extract AUC values in the right order for plotting.
auc_simple = [auc_results[(d, model_names[0])] for d in dataset_names]
auc_full   = [auc_results[(d, model_names[1])] for d in dataset_names]

# x positions for the two dataset groups, and bar width.
x_pos = np.arange(len(dataset_names))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))

bars_simple = ax.bar(x_pos - width / 2, auc_simple, width,
                     label='Simple model (3 regressors)', color='#2196F3', alpha=0.8)
bars_full   = ax.bar(x_pos + width / 2, auc_full,   width,
                     label='Full model (5 regressors)',   color='#FF9800', alpha=0.8)

# Print the AUC value above each bar so it is easy to read.
for bar in list(bars_simple) + list(bars_full):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=10)

ax.set_xticks(x_pos)
ax.set_xticklabels(dataset_names, fontsize=11)
ax.set_ylabel('AUC  (5-fold cross-validation)', fontsize=12)
ax.set_title(f'Model comparison via cross-validation  (n = {n} trials)', fontsize=12)
ax.set_ylim(0.5, 1.0)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Chance level')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

display(Markdown(r"""Are these results what you expected? Do you understand why they come out like this?"""))
display(Markdown(r"""At the beginning of the previous cell, one defines the size of the datasets
used in this exercise (which can be anything less than 5000). Try changing this parameter and run 
the two cells again to understand the effect of the size of the dataset on the problem of 
overfitting,"""))
